# Hands-on Lab: Detecting IoT Malware Behavior in Network Traffic

**Dataset:** CTU-IoT-Malware-Capture-1  

---

### Lab Overview

This lab explores IoT device behavior using the `CTU-IoT-Malware-Capture-1` dataset — real network traffic captured from an IoT device. The goal is to train both a Machine Learning (Random Forest) and a Deep Learning (Neural Network) model to classify traffic as **Benign** or **Malicious**.

### Lab Objectives
1. Preprocess the IoT network traffic dataset
2. Train a Random Forest classifier
3. Train a Neural Network classifier
4. Visualize DL training history
5. Compare model performance

## Step 1: Import Essential Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

from tensorflow import keras
from tensorflow.keras import layers

## Step 2: Load and Preprocess the Dataset

In [ ]:
data = pd.read_csv('CTU-IoT-Malware-Capture-1.csv')

print(data.head())

print("\nMissing values per column:")
print(data.isnull().sum())

## Step 3: Define Features and Target Variable

**Key decisions:**
- Target variable: `label` (Benign / Malicious)
- Drop `label` and `detailed-label` to prevent **data leakage** — the model must not see the answer during training
- Drop identifier columns (`uid`, `ts`, `id.orig_h`, `id.resp_h`) — these are session-specific and don't generalise to unseen traffic

In [ ]:
print("Columns in the dataset:", data.columns.tolist())

# Drop leakage columns, identifiers, and the target itself from features
drop_cols = ['label', 'detailed-label', 'uid', 'ts', 'id.orig_h', 'id.resp_h']
X = data.drop(columns=drop_cols)
y = data['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Features shape: {X.shape}, Target shape: {y.shape}")
print(f"\nLabel distribution:\n{y.value_counts()}")

## Step 4: Data Scaling and Dimensionality Reduction

Pipeline:
1. **StandardScaler** on numerical columns — normalises byte counts, packet counts, duration
2. **OneHotEncoder** on categorical columns — encodes protocol, service, connection state, history flags
3. **PCA** retaining 95% of variance — reduces dimensionality after one-hot expansion

In [ ]:
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numerical_cols   = X.select_dtypes(exclude=['object']).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:",   numerical_cols)

# Column transformer: scale numerics, one-hot encode categoricals
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                        numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'),  categorical_cols)
    ]
)

pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95))
])

X_train_transformed = pipeline.fit_transform(X_train)
X_test_transformed  = pipeline.transform(X_test)

print(f"\nTransformed training features shape: {X_train_transformed.shape}")
print(f"Transformed testing features shape:  {X_test_transformed.shape}")

## Step 5: Train and Evaluate the ML Model (Random Forest)

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train_transformed, y_train)

# Predict
y_pred_rf = rf_model.predict(X_test_transformed)

# Evaluate
print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

## Step 6: Train and Evaluate the DL Model (Neural Network)

Architecture: two hidden layers (64 → 32 ReLU) with a sigmoid output for binary classification.  
`LabelEncoder` maps string labels to 0/1 integers required by Keras.

In [ ]:
label_encoder   = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded  = label_encoder.transform(y_test)

dl_model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train_transformed.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(1,  activation='sigmoid')  # Binary output
])

dl_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = dl_model.fit(
    X_train_transformed,
    y_train_encoded,
    epochs=20,
    validation_split=0.2,
    verbose=1
)

# Predict and decode back to original labels
y_pred_dl     = (dl_model.predict(X_test_transformed) > 0.5).astype('int32')
y_pred_labels = label_encoder.inverse_transform(y_pred_dl.flatten())

print("\nDL Predictions:", y_pred_labels)

## Step 7: Visualise DL Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'],     label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[0].set_title('Model Accuracy')
axes[0].set_ylabel('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'],     label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Validation Loss')
axes[1].set_title('Model Loss')
axes[1].set_ylabel('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 8: Compare Model Performance

In [ ]:
# Encode test labels and RF predictions for unified comparison
y_test_encoded    = label_encoder.transform(y_test)
y_pred_rf_encoded = label_encoder.transform(y_pred_rf)

print("=== Random Forest Model Metrics ===")
print(classification_report(y_test_encoded, y_pred_rf_encoded,
                             target_names=label_encoder.classes_))

# DL: convert probabilities -> binary -> original labels
y_pred_dl_binary = (y_pred_dl.flatten() > 0.5).astype(int)
y_pred_dl_labels = label_encoder.inverse_transform(y_pred_dl_binary)

print("=== Deep Learning Model Metrics ===")
print(classification_report(y_test, y_pred_dl_labels,
                             target_names=label_encoder.classes_))

---

## Personal Analysis

### Dataset Limitation: Single-Class Labels

The provided CSV subset contains **153 rows, all labelled Benign**. This means both models are trained and evaluated on a single class — any classifier that predicts "Benign" for everything achieves 100% accuracy. The results therefore reflect the data limitation, not genuine detection capability.

The full CTU-IoT-Malware-Capture-1 dataset contains both benign and malicious traffic. In a production setting, this imbalance would require techniques such as:
- **SMOTE** (Synthetic Minority Over-sampling) to generate synthetic malicious samples
- **Class-weighted loss** in the neural network (`class_weight` parameter in `model.fit`)
- Evaluation focusing on **precision, recall, and F1** rather than accuracy

### Random Forest vs Neural Network on Small Datasets

Random Forest consistently outperforms neural networks on small tabular datasets like this one. The reasons are well understood:
- RF requires no label encoding or feature scaling to converge
- Neural networks need sufficient data to learn meaningful weight representations — 153 rows is well below that threshold
- RF is also more interpretable: `feature_importances_` would directly show which network features (byte counts, duration, connection state) are most predictive of malicious behaviour